In [ ]:
import numpy as np
from typing import Dict
from utils import utils
from utils.data_utils import load_metadata_and_map

In [ ]:
METADATA_FILE = "embeddings/id_metadata.json"
TENSOR_DIR = "embeddings/uuid_embeddings/embeddings/"

In [ ]:
# 1. Load Tensors
tensors = utils.load_tensor(TENSOR_DIR, num_workers=16)


In [ ]:
# 2. Load Metadata (Shared Logic)
df, artist_to_indices = load_metadata_and_map(METADATA_FILE, tensors)

In [ ]:
# 3. Pool Embeddings
print("Pooling embeddings (mean+max)...")
all_embeddings = utils.pool_loaded_tensor_dict(tensors=tensors, mode='mean+max')


In [ ]:
# Convert to standard numpy matrix
embedding_mat = np.array(list(all_embeddings.values()))

In [ ]:
# 4. Compute Centroids
print("Computing Artist Centroids...")
artist_centroids: Dict[str, np.ndarray] = {}

for artist_id, indices in artist_to_indices.items():
  # Select specific tracks for this artist
  track_embeds = embedding_mat[np.array(indices)]
  # Compute mean
  artist_centroids[artist_id] = track_embeds.mean(axis=0)


In [ ]:
from utils.data_utils import knn_search, get_artist_name
# 5. Search
QUERY_ARTIST_ID = '85de2a90-ea44-473f-af2f-2c22dcc99064'

print(f"Searching 10 nearest neighbors for {QUERY_ARTIST_ID} (Centroid)...")
if QUERY_ARTIST_ID not in artist_centroids:
  print(f"Error: Artist ID {QUERY_ARTIST_ID} not found.")

result_ids = knn_search(
  artist_centroids, 
  artist_centroids[QUERY_ARTIST_ID], 
  k=10, 
  backend=np
)

for res_id in result_ids:
  print(f"- {get_artist_name(df, res_id)}")
